In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **01_Read data**

### **1.0. Config**

In [ ]:
from pathlib import Path
import sys
import json
import itertools

import numpy as np
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/Legal_Sytem")
DATA_DIR = BASE_DIR / "01_Dataset/02_processed_data"
RETRIEVER_DIR = BASE_DIR / "02_Retriever"

BM25_OUTPUT_DIR = RETRIEVER_DIR / "02_Baseline/BM25/output"
FINETUNE_OUTPUT_DIR = RETRIEVER_DIR / "03_Finetuning/output"

OUTPUT_DIR = RETRIEVER_DIR / "04_Hybrid/output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DENSE_RUN_NAMES = [
    'vietnamese-bi-encoder__marginmse-cross_encoder',
    'bge-m3__marginmse-cross_encoder',
    'multilingual-e5-large__marginmse-cross_encoder',
    'Qwen3-Embedding-0.6B__marginmse-cross_encoder',
]

BM25_RUN_NAME = 'BM25'

ALPHA_GRID = [round(a, 2) for a in np.arange(0.0, 1.01, 0.05)]
RRF_K_GRID = [10, 20, 60, 100]
RRF_WEIGHT_GRID = [0.3, 0.5, 0.7]

PAPER_TOP_K = [1, 5, 10, 15, 20, 25, 30]

SELECTION_METRIC = 'mrr@1'

sys.path.append(str(RETRIEVER_DIR))

import importlib
import retriever_common


importlib.reload(retriever_common)
rc = retriever_common
rc.require_version(3)

print("TOP_K_LIST:", rc.TOP_K_LIST)

retriever_common v3 (cần >= 3)
TOP_K_LIST: [1, 3, 5, 10, 15, 20, 25, 30]


In [3]:
corpus = pd.read_parquet(DATA_DIR / 'corpus' / 'corpus.parquet')
query = pd.read_parquet(DATA_DIR / 'corpus' / 'queries.parquet')
qrels = pd.read_parquet(DATA_DIR / 'corpus' / 'qrels.parquet')

print(f"Corpus: {len(corpus):,} passage")

Corpus: 294 passage


### **1.1. Lọc các run_name có sẵn**

In [4]:
available_runs = sorted({path.name.split('__val_ranking')[0].split('__test_ranking')[0] for path in FINETUNE_OUTPUT_DIR.glob('*_ranking.parquet')})

print("Các run_name trong thư mục 03_Fine-tuning:")
for run_name in available_runs:
  marker = '  <-- đang chọn' if run_name in DENSE_RUN_NAMES else ''
  print(f" - {run_name}{marker}")

not_found = [name for name in DENSE_RUN_NAMES if name not in available_runs]

if not_found:
  print(f"\nCẢNH BÁO: {len(not_found)} run không có trong danh sách trên:")

  for name in not_found:
    print(f" - {name}")

  print("Sửa DENSE_RUN_NAMES ở mục 1 cho khớp.")

Các run_name trong thư mục 03_Fine-tuning:
 - Qwen3-Embedding-0.6B__marginmse-cross_encoder  <-- đang chọn
 - Qwen3-Embedding-4B__marginmse-cross_encoder
 - bge-m3__marginmse-cross_encoder  <-- đang chọn
 - multilingual-e5-large__marginmse-cross_encoder  <-- đang chọn
 - vietnamese-bi-encoder__marginmse-cross_encoder  <-- đang chọn


In [5]:
def load_ranking(directory: Path, run_name: str, split_name: str) -> pd.DataFrame:
  if run_name == BM25_RUN_NAME:
    path = directory / f'{split_name}_ranking.parquet'
  else:
    path = directory / f'{run_name}__{split_name}_ranking.parquet'

  if not path.exists():
    raise FileNotFoundError(f"Không tìm thấy {path}")

  return pd.read_parquet(path)[['query_id', 'doc_id', 'rank', 'score']]

In [6]:
bm25_ranking = {
    split_name: load_ranking(BM25_OUTPUT_DIR, BM25_RUN_NAME, split_name)
    for split_name in ['val', 'test']
}

In [8]:
dense_ranking = {}
missing_runs = []

for run_name in DENSE_RUN_NAMES:
  try:
    dense_ranking[run_name] = {
        split_name: load_ranking(FINETUNE_OUTPUT_DIR, run_name, split_name) for split_name in ['val', 'test']
    }
  except FileNotFoundError as error:
    missing_runs.append(run_name)
    print(f"Bỏ qua '{run_name}': {error}")

assert dense_ranking, "Không nạp được run dense nào. Kiểm tra DENSE_RUN_NAMES ở mục 1."

In [9]:
print(f"\nBM25: {len(bm25_ranking['val']):,} dòng val, độ sâu {bm25_ranking['val']['rank'].max()}")

for run_name, rankings in dense_ranking.items():
  print(f"{run_name}: {len(rankings['val']):,} dòng val, độ sâu {rankings['val']['rank'].max()}")

if missing_runs:
  print(f"\n{len(missing_runs)} run không tìm thấy — sửa DENSE_RUN_NAMES rồi chạy lại.")


BM25: 46,650 dòng val, độ sâu 50
vietnamese-bi-encoder__marginmse-cross_encoder: 46,650 dòng val, độ sâu 50
bge-m3__marginmse-cross_encoder: 46,650 dòng val, độ sâu 50
multilingual-e5-large__marginmse-cross_encoder: 46,650 dòng val, độ sâu 50
Qwen3-Embedding-0.6B__marginmse-cross_encoder: 46,650 dòng val, độ sâu 50


### 1.2. Check lai query trong BM25 va dense ranking co dong nhat

In [10]:
for split_name in ['val', 'test']:
  expected_query_ids = set(query.loc[query['split'] == split_name, 'query_id'])
  bm25_query_ids = set(bm25_ranking[split_name]['query_id'])

  assert bm25_query_ids == expected_query_ids, (f"BM25 {split_name}: tập câu hỏi không khớp queries.parquet — có thể file ranking được sinh trước khi lọc rò rỉ.")

  for run_name, rankings in dense_ranking.items():
    dense_query_ids = set(rankings[split_name]['query_id'])

    assert dense_query_ids == expected_query_ids, (
        f"{run_name} / {split_name}: tập câu hỏi lệch so với queries.parquet (thiếu {len(expected_query_ids - dense_query_ids)}, thừa {len(dense_query_ids - expected_query_ids)})")

  print(f"{split_name}: {len(expected_query_ids):,} câu hỏi, mọi hệ thống khớp")

val: 933 câu hỏi, mọi hệ thống khớp
test: 958 câu hỏi, mọi hệ thống khớp


## **02. Candidates**

In [ ]:
MISSING_RANK = 10 ** 6   


def build_candidate_table(bm25_ranking: pd.DataFrame, dense_ranking: pd.DataFrame) -> pd.DataFrame:
  merged = pd.merge(
      bm25_ranking.rename(columns={'rank': 'rank_bm25', 'score': 'score_bm25'}),
      dense_ranking.rename(columns={'rank': 'rank_dense', 'score': 'score_dense'}),
      on=['query_id', 'doc_id'], how='outer',
  )

  merged['rank_bm25'] = merged['rank_bm25'].fillna(MISSING_RANK).astype(int)
  merged['rank_dense'] = merged['rank_dense'].fillna(MISSING_RANK).astype(int)

  for column in ['score_bm25', 'score_dense']:
    per_query_min = merged.groupby('query_id')[column].transform('min')
    merged[column] = merged[column].fillna(per_query_min)

  for column in ['score_bm25', 'score_dense']:
    grouped = merged.groupby('query_id')[column]

    minimum = grouped.transform('min')
    maximum = grouped.transform('max')
    spread = (maximum - minimum).replace(0, np.nan)

    merged[f'{column}_minmax'] = ((merged[column] - minimum) / spread).fillna(0.5)

    mean = grouped.transform('mean')
    std = grouped.transform('std').replace(0, np.nan)

    merged[f'{column}_z'] = ((merged[column] - mean) / std).fillna(0.0)

  return merged

In [12]:
candidate_table = {
    run_name: {
        split_name: build_candidate_table(bm25_ranking[split_name], dense_ranking[run_name][split_name]) for split_name in ['val', 'test']
    } for run_name in dense_ranking
}

for run_name, tables in candidate_table.items():
  table = tables['val']
  candidates_per_query = table.groupby('query_id').size()

  print(f"{run_name}: trung bình {candidates_per_query.mean():.1f} ứng viên/câu hỏi (chỉ BM25: {int((table['rank_dense'] == MISSING_RANK).sum()):,}, chỉ dense: {int((table['rank_bm25'] == MISSING_RANK).sum()):,})")

vietnamese-bi-encoder__marginmse-cross_encoder: trung bình 76.2 ứng viên/câu hỏi (chỉ BM25: 24,442, chỉ dense: 24,442)
bge-m3__marginmse-cross_encoder: trung bình 84.4 ứng viên/câu hỏi (chỉ BM25: 32,131, chỉ dense: 32,131)
multilingual-e5-large__marginmse-cross_encoder: trung bình 84.0 ứng viên/câu hỏi (chỉ BM25: 31,756, chỉ dense: 31,756)
Qwen3-Embedding-0.6B__marginmse-cross_encoder: trung bình 80.2 ứng viên/câu hỏi (chỉ BM25: 28,223, chỉ dense: 28,223)


## **03. Hybrib**

In [ ]:
def fuse(table: pd.DataFrame, method: str, **parameters) -> pd.Series:
  if method == 'rrf':
    k = parameters['k'] # Hằng số bù (thường mặc định là 60)
    weight = parameters['weight']   # trọng số cho BM25

    return (weight / (k + table['rank_bm25'])  + (1.0 - weight) / (k + table['rank_dense']))

  if method == 'weighted_minmax':
    alpha = parameters['alpha']

    return alpha * table['score_bm25_minmax'] + (1.0 - alpha) * table['score_dense_minmax']

  if method == 'weighted_zscore':
    alpha = parameters['alpha']

    return alpha * table['score_bm25_z'] + (1.0 - alpha) * table['score_dense_z']

  if method == 'multiplication':
    return (table['score_bm25_minmax'] + 1e-6) * (table['score_dense_minmax'] + 1e-6)

  if method == 'paper':
    alpha = parameters['alpha']

    return alpha * table['score_bm25'] + (1.0 - alpha) * table['score_dense_minmax']

  raise ValueError(f"Phương pháp không hợp lệ: {method}")

In [ ]:
def rank_and_evaluate(table: pd.DataFrame, fused_score: pd.Series, split_name: str, qrels_map: dict):
  scored = table[['query_id', 'doc_id']].copy()
  scored['score'] = fused_score.to_numpy()

  scored = scored.sort_values(['query_id', 'score'], ascending=[True, False], kind='mergesort')

  scored['rank'] = scored.groupby('query_id').cumcount() + 1
  scored['split'] = split_name

  scored = scored[scored['rank'] <= rc.RANKING_DEPTH].reset_index(drop=True)

  per_query = rc.evaluation(scored, qrels_map)

  return scored, per_query

In [15]:
qrels_map_by_split = {
    split_name: rc.build_qrels_map(qrels[qrels['split'] == split_name])
    for split_name in ['val', 'test']
}

## **04. Val parameter**

In [16]:
configurations = []

for k, weight in itertools.product(RRF_K_GRID, RRF_WEIGHT_GRID):
  configurations.append({'method': 'rrf', 'k': k, 'weight': weight})

for alpha in ALPHA_GRID:
  configurations.append({'method': 'weighted_minmax', 'alpha': alpha})
  configurations.append({'method': 'weighted_zscore', 'alpha': alpha})
  configurations.append({'method': 'paper', 'alpha': alpha})

configurations.append({'method': 'multiplication'})

print(f"Số cấu hình cần quét: {len(configurations)}")

Số cấu hình cần quét: 76


In [17]:
def sweep_one_model(run_name: str) -> pd.DataFrame:

  validation_table = candidate_table[run_name]['val']
  records = []

  for configuration in configurations:
    parameters = {key: value for key, value in configuration.items() if key != 'method'}

    fuse_scores = fuse(validation_table, configuration['method'], **parameters)

    _, per_query = rank_and_evaluate(validation_table, fuse_scores, 'val', qrels_map_by_split['val'])

    summary = rc.summarize(per_query)

    records.append({
        'dense_run': run_name,
        'method': configuration['method'],
        **parameters,
        'P@1': round(summary['p@1'] * 100, 2),
        'P@5': round(summary['p@5'] * 100, 2),
        'P@10': round(summary['p@10'] * 100, 2),
        'P@15': round(summary['p@15'] * 100, 2),
        'P@20': round(summary['p@20'] * 100, 2),
        'MRR@1': round(summary['mrr@1'] * 100, 2),
        'MRR@5': round(summary['mrr@5'] * 100, 2),
        'MRR@10': round(summary['mrr@10'] * 100, 2),
        'MRR@15': round(summary['mrr@15'] * 100, 2),
        'MRR@20': round(summary['mrr@20'] * 100, 2),
        'nDCG@1': round(summary['ndcg@1'] * 100, 2),
        'nDCG@5': round(summary['ndcg@5'] * 100, 2),
        'nDCG@10': round(summary['ndcg@10'] * 100, 2),
        'nDCG@15': round(summary['ndcg@15'] * 100, 2),
        'nDCG@20': round(summary['ndcg@20'] * 100, 2),
        'Recall@1': round(summary['recall@1'] * 100, 2),
        'Rrecall@5': round(summary['recall@5'] * 100, 2),
        'Recall@10': round(summary['recall@10'] * 100, 2),
        'Recall@15': round(summary['recall@15'] * 100, 2),
        'Recall@20': round(summary['recall@20'] * 100, 2),
    })

  return pd.DataFrame(records)

In [18]:
sweep_dataframe = pd.concat(
    [sweep_one_model(run_name) for run_name in dense_ranking], ignore_index=True
)

# SELECTION_COLUMN = {'p@1': 'P@1', 'p@5': 'P@5', 'p@10': 'P@10', 'p@15': 'P@15', 'p@20': 'P@20', 'mrr@1': 'MRR@1', 'mrr@5': 'MRR@5',  'mrr@10': 'MRR@10', 'mrr@15': 'MRR@15', 'mrr@20': 'MRR@20', 'nDCG@1': 'nDCG@1', 'nDCG@5': 'nDCG@5', 'nDCG@10': 'nDCG@10'}[SELECTION_METRIC]

sweep_dataframe.to_csv(OUTPUT_DIR / 'hybrid_sweep_all_models.csv', index=False, encoding='utf-8-sig')

print(f"Đã quét {len(sweep_dataframe):,} cấu hình: ({len(configurations)} x {len(dense_ranking)} model). Chọn theo {sweep_dataframe.columns}.")

Đã quét 304 cấu hình: (76 x 4 model). Chọn theo Index(['dense_run', 'method', 'k', 'weight', 'P@1', 'P@5', 'P@10', 'P@15',
       'P@20', 'MRR@1', 'MRR@5', 'MRR@10', 'MRR@15', 'MRR@20', 'nDCG@1',
       'nDCG@5', 'nDCG@10', 'nDCG@15', 'nDCG@20', 'Recall@1', 'Rrecall@5',
       'Recall@10', 'Recall@15', 'Recall@20', 'alpha'],
      dtype='object').


In [19]:
SELECTION_COLUMN = {'p@1': 'P@1', 'p@5': 'P@5', 'p@10': 'P@10', 'p@15': 'P@15', 'p@20': 'P@20', 'mrr@1': 'MRR@1', 'mrr@5': 'MRR@5', 'mrr@10': 'MRR@10', 'mrr@15': 'MRR@15', 'mrr@20': 'MRR@20', 'nDCG@1': 'nDCG@1', 'nDCG@5': 'nDCG@5', 'nDCG@10': 'nDCG@10', 'recall@1': 'Recall@1', 'recall@5': 'Recall@5', 'recall@10': 'Recall@10', 'recall@15': 'Recall@15', 'recall@20': 'Recall@20'}[SELECTION_METRIC]

best_per_model_method = (
    sweep_dataframe
    .sort_values(by=SELECTION_COLUMN, ascending=False)
    .groupby(['dense_run', 'method'], as_index=False)
    .first()
    .sort_values(by=['dense_run', SELECTION_COLUMN], ascending=[True, False])
    .reset_index(drop=True)
)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
display(best_per_model_method)

,dense_run,method,k,weight,P@1,P@5,P@10,P@15,P@20,MRR@1,MRR@5,MRR@10,MRR@15,MRR@20,nDCG@1,nDCG@5,nDCG@10,nDCG@15,nDCG@20,Recall@1,Rrecall@5,Recall@10,Recall@15,Recall@20,alpha
0,Qwen3-Embedding-0.6B__marginmse-cross_encoder,paper,NaN,NaN,77.06,96.14,98.29,98.93,99.14,77.06,85.22,85.50,85.55,85.57,77.06,88.00,88.69,88.86,88.91,77.06,96.14,98.29,98.93,99.14,0.15
1,Qwen3-Embedding-0.6B__marginmse-cross_encoder,weighted_zscore,NaN,NaN,76.74,95.28,98.39,98.82,99.25,76.74,84.67,85.11,85.14,85.16,76.74,87.38,88.40,88.51,88.61,76.74,95.28,98.39,98.82,99.25,0.55
2,Qwen3-Embedding-0.6B__marginmse-cross_encoder,weighted_minmax,NaN,NaN,76.31,95.71,98.29,98.71,99.36,76.31,84.61,84.96,85.00,85.03,76.31,87.44,88.28,88.40,88.55,76.31,95.71,98.29,98.71,99.36,0.55
3,Qwen3-Embedding-0.6B__marginmse-cross_encoder,multiplication,NaN,NaN,74.71,92.93,95.71,97.75,98.71,74.71,82.55,82.93,83.09,83.15,74.71,85.19,86.10,86.64,86.87,74.71,92.93,95.71,97.75,98.71,NaN
4,Qwen3-Embedding-0.6B__marginmse-cross_encoder,rrf,60.0,0.7,72.99,93.25,96.03,97.53,98.18,72.99,81.40,81.76,81.88,81.92,72.99,84.40,85.29,85.69,85.84,72.99,93.25,96.03,97.53,98.18,NaN
5,bge-m3__marginmse-cross_encoder,paper,NaN,NaN,79.85,94.75,96.68,97.64,98.50,79.85,86.21,86.47,86.54,86.59,79.85,88.38,89.00,89.26,89.46,79.85,94.75,96.68,97.64,98.50,0.05
6,bge-m3__marginmse-cross_encoder,weighted_zscore,NaN,NaN,79.53,96.25,97.64,98.61,99.14,79.53,86.70,86.89,86.96,86.99,79.53,89.13,89.58,89.84,89.96,79.53,96.25,97.64,98.61,99.14,0.45
7,bge-m3__marginmse-cross_encoder,weighted_minmax,NaN,NaN,79.10,95.93,97.53,98.50,98.93,79.10,86.33,86.56,86.63,86.66,79.10,88.77,89.31,89.56,89.66,79.10,95.93,97.53,98.50,98.93,0.45
8,bge-m3__marginmse-cross_encoder,multiplication,NaN,NaN,77.81,94.53,97.21,98.18,98.71,77.81,85.17,85.52,85.60,85.63,77.81,87.56,88.43,88.69,88.81,77.81,94.53,97.21,98.18,98.71,NaN
9,bge-m3__marginmse-cross_encoder,rrf,10.0,0.5,75.03,96.14,97.75,98.82,99.14,75.03,84.07,84.29,84.38,84.40,75.03,87.14,87.67,87.96,88.03,75.03,96.14,97.75,98.82,99.14,NaN


In [20]:
best_per_model = (best_per_model_method.sort_values(SELECTION_COLUMN, ascending=False).groupby('dense_run', as_index=False).first().sort_values(SELECTION_COLUMN, ascending=False).reset_index(drop=True))

display(best_per_model[['dense_run', 'method', 'alpha', 'k', 'weight', 'P@1', 'P@5', 'P@10', 'MRR@1', 'MRR@5', 'MRR@10', 'nDCG@1', 'nDCG@5', 'nDCG@10']])

,dense_run,method,alpha,k,weight,P@1,P@5,P@10,MRR@1,MRR@5,MRR@10,nDCG@1,nDCG@5,nDCG@10
0,multilingual-e5-large__marginmse-cross_encoder,weighted_zscore,0.45,10.0,0.5,82.64,96.89,98.18,82.64,88.72,88.88,82.64,90.80,91.20
1,bge-m3__marginmse-cross_encoder,paper,0.05,10.0,0.5,79.85,94.75,96.68,79.85,86.21,86.47,79.85,88.38,89.00
2,vietnamese-bi-encoder__marginmse-cross_encoder,paper,0.10,10.0,0.7,78.99,97.11,99.14,78.99,86.74,87.03,78.99,89.38,90.05
3,Qwen3-Embedding-0.6B__marginmse-cross_encoder,paper,0.15,60.0,0.7,77.06,96.14,98.29,77.06,85.22,85.50,77.06,88.00,88.69


In [21]:
endpoint_check = sweep_dataframe[(sweep_dataframe['method'] == 'weighted_minmax') & (sweep_dataframe['alpha'].isin([0.0, 1.0]))].pivot(index='dense_run', columns='alpha', values='P@1')

endpoint_check.columns = ['alpha=0 (dense thuần)', 'alpha=1 (BM25 thuần)']
display(endpoint_check)

bm25_only_values = endpoint_check['alpha=1 (BM25 thuần)'].round(2).unique()

if len(bm25_only_values) == 1:
  print(f"alpha=1 nhất quán ở mọi model: {bm25_only_values[0]:.2f} — bảng ứng viên hợp lệ.")
else:
  print(f"CẢNH BÁO: alpha=1 khác nhau giữa các model {bm25_only_values} — kiểm tra lại.")

,alpha=0 (dense thuần),alpha=1 (BM25 thuần)
dense_run,,
Qwen3-Embedding-0.6B__marginmse-cross_encoder,62.70,71.28
bge-m3__marginmse-cross_encoder,66.88,71.28
multilingual-e5-large__marginmse-cross_encoder,74.28,71.28
vietnamese-bi-encoder__marginmse-cross_encoder,69.77,71.28


alpha=1 nhất quán ở mọi model: 71.28 — bảng ứng viên hợp lệ.


In [22]:
bm25_val_per_query = rc.evaluation(bm25_ranking['val'], qrels_map_by_split['val'])

headroom_records = []

for run_name, rankings in dense_ranking.items():
  dense_val_per_query = rc.evaluation(rankings['val'], qrels_map_by_split['val'])

  merged = pd.merge(
      bm25_val_per_query[['query_id', 'p@1']],
      dense_val_per_query[['query_id', 'p@1']],
      on='query_id', suffixes=('_bm25', '_dense'),
  )

  bm25_correct = merged['p@1_bm25'] > 0
  dense_correct = merged['p@1_dense'] > 0

  candidate_pairs = set(zip(candidate_table[run_name]['val']['query_id'],candidate_table[run_name]['val']['doc_id']))

  ceiling = float(np.mean([any((query_id, doc_id) in candidate_pairs for doc_id in gold_doc_ids) for query_id, gold_doc_ids in qrels_map_by_split['val'].items()]))

  headroom_records.append({
      'dense_run': run_name,
      'BM25 P@1': round(bm25_correct.mean() * 100, 2),
      'dense P@1': round(dense_correct.mean() * 100, 2),
      'cả hai đúng': int((bm25_correct & dense_correct).sum()),
      'chỉ BM25': int((bm25_correct & ~dense_correct).sum()),
      'chỉ dense': int((~bm25_correct & dense_correct).sum()),
      'cả hai sai': int((~bm25_correct & ~dense_correct).sum()),
      'trần fusion': round(ceiling * 100, 2),
  })

display(pd.DataFrame(headroom_records))

,dense_run,BM25 P@1,dense P@1,cả hai đúng,chỉ BM25,chỉ dense,cả hai sai,trần fusion
0,vietnamese-bi-encoder__marginmse-cross_encoder,71.28,69.77,515,150,136,132,100.0
1,bge-m3__marginmse-cross_encoder,71.28,66.88,481,184,143,125,100.0
2,multilingual-e5-large__marginmse-cross_encoder,71.28,74.28,535,130,158,110,100.0
3,Qwen3-Embedding-0.6B__marginmse-cross_encoder,71.28,62.70,467,198,118,150,100.0


## **06. Evaluate test**

In [23]:
def configuration_from_row(row) -> tuple:
  parameters = {
      key: row[key] for key in ['alpha', 'k', 'weight']
      if key in row and pd.notna(row[key])
  }

  if 'k' in parameters:
    parameters['k'] = int(parameters['k'])

  return row['method'], parameters


hybrid_results = {}

for record in best_per_model.to_dict('records'):
  run_name = record['dense_run']
  method, parameters = configuration_from_row(record)

  ranking_by_split = {}
  per_query_by_split = {}

  for split_name in ['val', 'test']:
    table = candidate_table[run_name][split_name]

    scored, per_query = rank_and_evaluate(table, fuse(table, method, **parameters), split_name, qrels_map_by_split[split_name])

    rc.assert_metrics_are_sane(per_query)

    ranking_by_split[split_name] = scored
    per_query_by_split[split_name] = per_query

  hybrid_results[run_name] = {
      'method': method,
      'parameters': parameters,
      'ranking': ranking_by_split,
      'per_query': per_query_by_split,
  }

  summary = rc.summarize(per_query_by_split['test'])

  print(f"{run_name}")
  print(f"  - {method} {parameters}")
  print(f"  - [test]  + {"  ".join(f"P@{k}={summary[f'p@{k}'] * 100:.2f}" for k in [1, 5, 10])}  MRR@1={summary['mrr@1'] * 100:.2f}")

multilingual-e5-large__marginmse-cross_encoder
  - weighted_zscore {'alpha': 0.45, 'k': 10, 'weight': 0.5}
  - [test]  + P@1=81.52  P@5=96.45  P@10=97.91  MRR@1=81.52
bge-m3__marginmse-cross_encoder
  - paper {'alpha': 0.05, 'k': 10, 'weight': 0.5}
  - [test]  + P@1=77.97  P@5=94.26  P@10=96.56  MRR@1=77.97
vietnamese-bi-encoder__marginmse-cross_encoder
  - paper {'alpha': 0.1, 'k': 10, 'weight': 0.7}
  - [test]  + P@1=75.37  P@5=95.30  P@10=98.23  MRR@1=75.37
Qwen3-Embedding-0.6B__marginmse-cross_encoder
  - paper {'alpha': 0.15, 'k': 60, 'weight': 0.7}
  - [test]  + P@1=73.80  P@5=93.84  P@10=97.29  MRR@1=73.80


In [24]:
test_records = []

bm25_test_per_query = rc.evaluation(bm25_ranking['test'], qrels_map_by_split['test'])
bm25_test_summary = rc.summarize(bm25_test_per_query)

test_records.append({
    'hệ_thống': 'BM25 (thuần)',
    'cấu_hình': f"k1={BM25_RUN_NAME}",
    **{f'P@{k}': round(bm25_test_summary[f'p@{k}'] * 100, 2) for k in PAPER_TOP_K},
    'MRR@1': round(bm25_test_summary['mrr@1'] * 100, 2),
})

for run_name, result in hybrid_results.items():
  dense_summary = rc.summarize(rc.evaluation(dense_ranking[run_name]['test'], qrels_map_by_split['test']))

  test_records.append({
      'hệ_thống': f'{run_name} (thuần)',
      'cấu_hình': '—',
      **{f'P@{k}': round(dense_summary[f'p@{k}'] * 100, 2) for k in PAPER_TOP_K},
      'MRR@1': round(dense_summary['mrr@1'] * 100, 2),
  })

  hybrid_summary = rc.summarize(result['per_query']['test'])

  test_records.append({
      'hệ_thống': f'BM25 + {run_name}',
      'cấu_hình': f"{result['method']} {result['parameters']}",
      **{f'P@{k}': round(hybrid_summary[f'p@{k}'] * 100, 2) for k in PAPER_TOP_K},
      'MRR@1': round(hybrid_summary['mrr@1'] * 100, 2),
  })

test_table = pd.DataFrame(test_records)

display(test_table)

test_table.to_csv(OUTPUT_DIR / 'hybrid_test_table.csv', index=False, encoding='utf-8-sig')

,hệ_thống,cấu_hình,P@1,P@5,P@10,P@15,P@20,P@25,P@30,MRR@1
0,BM25 (thuần),k1=BM25,66.91,89.04,92.90,94.78,95.62,96.24,96.56,66.91
1,multilingual-e5-large__marginmse-cross_encoder...,—,73.38,92.07,95.41,96.24,96.87,97.18,97.49,73.38
2,BM25 + multilingual-e5-large__marginmse-cross_...,"weighted_zscore {'alpha': 0.45, 'k': 10, 'weig...",81.52,96.45,97.91,98.54,98.75,98.85,98.96,81.52
3,bge-m3__marginmse-cross_encoder (thuần),—,67.85,89.25,93.84,94.68,96.14,97.18,97.70,67.85
4,BM25 + bge-m3__marginmse-cross_encoder,"paper {'alpha': 0.05, 'k': 10, 'weight': 0.5}",77.97,94.26,96.56,97.81,98.43,98.96,99.06,77.97
5,vietnamese-bi-encoder__marginmse-cross_encoder...,—,69.10,94.26,97.49,98.33,98.54,98.85,99.06,69.10
6,BM25 + vietnamese-bi-encoder__marginmse-cross_...,"paper {'alpha': 0.1, 'k': 10, 'weight': 0.7}",75.37,95.30,98.23,98.64,98.96,99.06,99.27,75.37
7,Qwen3-Embedding-0.6B__marginmse-cross_encoder ...,—,59.29,88.20,93.11,95.62,97.08,97.60,98.12,59.29
8,BM25 + Qwen3-Embedding-0.6B__marginmse-cross_e...,"paper {'alpha': 0.15, 'k': 60, 'weight': 0.7}",73.80,93.84,97.29,98.54,99.27,99.37,99.48,73.80


In [28]:
# Lưu file ranking.parquet của phương pháp tối ưu nhất là BM25 + multilingual-e5-large__marginmse-cross-encoder

best_run_name = 'multilingual-e5-large__marginmse-cross_encoder'
best_ranking_df = hybrid_results[best_run_name]['ranking']['test']

output_file_path = OUTPUT_DIR / 'best_hybrid_ranking.parquet'
best_ranking_df.to_parquet(output_file_path, index=False)

print(f"Đã lưu kết quả ranking của phương pháp tối ưu nhất vào: {output_file_path}")

Đã lưu kết quả ranking của phương pháp tối ưu nhất vào: /content/drive/MyDrive/Legal_Sytem/02_Retriever/04_Hybrid/output/best_hybrid_ranking.parquet


In [26]:
best_run_name = 'multilingual-e5-large__marginmse-cross_encoder'
best_ranking_df = hybrid_results[best_run_name]['ranking']['test']

In [27]:
best_ranking_df.head(2)

,query_id,doc_id,score,rank,split
0,QUERY_003672539f300718352e,DOC_a62c3419fe62a6dc6bd6,5.554094,1,test
1,QUERY_003672539f300718352e,DOC_5c55772aeb40577970a0,3.109321,2,test
